<a href="https://colab.research.google.com/github/pandahsu849/Programing_Languages/blob/main/HW1_%E6%97%A5%E5%B8%B8%E6%94%AF%E5%87%BA%E9%80%9F%E7%AE%97%E8%88%87%E5%88%86%E6%94%A4_Gradio_Part2%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#日常支出速算與分攤（作業一）
- 目標：從 Sheet 讀「消費紀錄」→ 計總額/分類小計/AA 分攤 → 寫回 Sheet Summary 分頁。
- AI 點子（可選）：請模型總結本週花錢習慣與建議（例如「外食過多」）。
- Sheet 欄位：date, category, item, amount, payer

GoogleSheet: https://docs.google.com/spreadsheets/d/1CQZDzhSqoIbR8JjExDr9eTq0qDzP6vQvUp9JAs4Jmic/edit?usp=sharing

In [11]:
import gradio as gr
import pandas as pd
import datetime
from datetime import timedelta, timezone
import gspread
import datetime
import plotly.express as px
from google.colab import auth
from google.auth import default

In [2]:
SHEET_URL = "https://docs.google.com/spreadsheets/d/1CQZDzhSqoIbR8JjExDr9eTq0qDzP6vQvUp9JAs4Jmic/edit?usp=sharing"
WORKSHEET_NAME = "工作表2"

REQUIRED_COLUMNS = ["日期", "時間", "分類", "品項", "金額", "付款人", "地點"]

_auth_done = False
_gc = None
_ws = None

In [15]:
def _ensure_auth_and_ws():
    global _auth_done, _gc, _ws
    if not _auth_done:
        # Colab 使用者授權
        auth.authenticate_user()
        creds, _ = default()
        _gc = gspread.authorize(creds)
        _auth_done = True
    if _ws is None:
        gs = _gc.open_by_url(SHEET_URL)
        _ws = gs.worksheet(WORKSHEET_NAME)
        # 確保欄位完整
        _ensure_headers()
    return _ws

def _ensure_headers():
    """確保表頭包含 REQUIRED_COLUMNS；若空表或缺欄，會補齊。"""
    rows = _ws.get_all_values()
    if not rows:
        _ws.append_row(REQUIRED_COLUMNS, value_input_option="USER_ENTERED")
        return
    header = rows[0]
    if header != REQUIRED_COLUMNS:
        # 合併既有欄與必需欄，並以 REQUIRED_COLUMNS 為主順序
        existing = {h: idx for idx, h in enumerate(header)}
        # 若第一列不是必需欄，重寫表頭並搬移資料欄位（簡化處理：補欄位）
        _ws.update('1:1', [REQUIRED_COLUMNS])

        # 若舊資料有相同欄位名，保留；沒有的欄位留空
        if len(rows) > 1:
            new_rows = []
            for r in rows[1:]:
                mapping = {col: (r[existing[col]] if col in existing and existing[col] < len(r) else "")
                           for col in REQUIRED_COLUMNS}
                new_rows.append([mapping[c] for c in REQUIRED_COLUMNS])
            # 先清掉舊內容只留表頭
            _ws.resize(rows=1)
            # 再補回資料
            _ws.append_rows(new_rows, value_input_option="USER_ENTERED")

def _read_df():
    ws = _ensure_auth_and_ws()
    values = ws.get_all_values()
    if not values:
        return pd.DataFrame(columns=REQUIRED_COLUMNS)
    df = pd.DataFrame(values[1:], columns=values[0])
    # 型別整理
    if "金額" in df.columns:
        df["金額"] = pd.to_numeric(df["金額"], errors="coerce").fillna(0.0)
    return df

def add_expense(date_str, time_str, category, item, amount, payer, location, note):
    try:
        tz = timezone(timedelta(hours=8))
        now = datetime.datetime.now(tz)
        # 日期驗證
        if not date_str:
            date_str = now.strftime("%Y-%m-%d")

        # 時間驗證
        time_str = (time_str or "").strip()
        if not time_str:
            time_str = now.strftime("%H:%M")

        # 類別/品項/付款人預設
        category = (category or "未填").strip()
        item = (item or "未填").strip()
        payer = (payer or "匿名").strip()

        # 允許空白備註
        note = (note or "").strip()

        # 金額
        try:
            amount_val = float(amount)
        except:
            return "金額格式錯誤，請輸入數字", None, None, None

        ws = _ensure_auth_and_ws()
        # 直接 append 一列
        ws.append_row(
            [date_str, time_str, category, item, amount_val, payer, location, note],
            value_input_option="USER_ENTERED"
        )
        msg = f"✅ 已新增：{date_str} {time_str}｜{category}｜{item}｜{amount_val}｜{payer}｜{location}｜{note}"
        # 回傳即時摘要
        df = _read_df()
        cat, settle = _make_summary_tables(df)
        total = float(df["金額"].sum()) if not df.empty else 0.0
        return msg, total, cat, settle
    except Exception as e:
        return f"❌ 新增失敗：{e}", None, None, None

def refresh_summary(budget_limit_val):
    try:
        df = _read_df()
        if df.empty:
            return "目前沒有資料", 0.0, pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), ""

        # --- 只計算「本月」資料 ---
        # 確保台灣時區
        tz = timezone(timedelta(hours=8))
        this_month_str = datetime.datetime.now(tz).strftime("%Y-%m")

        # 過濾日期欄位開頭為今年今月的資料 (例如 "2024-05")
        df_this_month = df[df['日期'].str.startswith(this_month_str)].copy()

        # 計算本月總額
        total_this_month = float(df_this_month["金額"].sum()) if not df_this_month.empty else 0.0

        # 製作摘要表格 (使用全部資料或本月資料，建議摘要可以用本月的)
        by_cat, settle = _make_summary_tables(df_this_month)

        # 呼叫剛才寫好的進度條生成器
        progress_html = get_budget_progress(total_this_month, budget_limit_val)

        return f"✅ 已更新 {this_month_str} 彙總", total_this_month, by_cat, settle, df_this_month.tail(5), progress_html

    except Exception as e:
        return f"❌ 讀取失敗：{e}", 0.0, pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), ""

#圓餅圖
def _make_summary_tables(df: pd.DataFrame):
    # 分類小計
    if "分類" in df.columns:
        by_cat = df.groupby("分類", as_index=False)["金額"].sum().sort_values("金額", ascending=False)
    else:
        by_cat = pd.DataFrame(columns=["分類", "金額"])

    # 付款人結算（AA 分攤）
    if "付款人" in df.columns and df["付款人"].notna().any():
        payers = df["付款人"].replace("", "匿名")
        unique_payers = sorted([p for p in payers.unique() if p])
        total = df["金額"].sum()
        n = max(len(unique_payers), 1)
        aa_share = total / n
        paid_by = df.groupby("付款人", as_index=False)["金額"].sum().rename(columns={"金額": "實付"})
        # 確保每位付款人都有列
        all_rows = pd.DataFrame({"付款人": unique_payers}).merge(paid_by, on="付款人", how="left").fillna({"實付": 0.0})
        all_rows["應付(AA)"] = aa_share
        all_rows["差額(實付-應付)"] = all_rows["實付"] - all_rows["應付(AA)"]
        settle = all_rows.sort_values("差額(實付-應付)", ascending=False)
    else:
        settle = pd.DataFrame(columns=["付款人", "實付", "應付(AA)", "差額(實付-應付)"])
    return by_cat, settle

def _view_all_with_chart():
    try:
        df = _read_df()
        if df.empty:
            # 如果沒資料，回傳空圖表與空表格
            return None, pd.DataFrame(columns=["日期", "時間", "分類", "品項", "金額", "付款人", "地點", "備註"])

        # --- 製作圓餅圖邏輯 ---
        # 1. 依照分類加總金額
        chart_data = df.groupby("分類")["金額"].sum().reset_index()

        # 2. 使用 Plotly 繪圖
        fig = px.pie(
            chart_data,
            values='金額',
            names='分類',
            hole=0.4, # 變成環狀圖比較好看
            title="💰 各類別支出比例分析",
            color_discrete_sequence=px.colors.qualitative.Pastel # 使用粉彩色系
        )

        # 優化圖表佈局（讓它在手機/網頁上更好看）
        fig.update_layout(
            margin=dict(t=50, b=20, l=20, r=20),
            legend=dict(orientation="h", yanchor="bottom", y=-0.2, xanchor="center", x=0.5)
        )

        # 3. 資料排序：最新的日期排在最上面
        df_sorted = df.sort_values(by=["日期"], ascending=False)

        return fig, df_sorted

    except Exception as e:
        # 發生錯誤時，圖表回傳 None，表格顯示錯誤訊息
        return None, pd.DataFrame({"錯誤": [str(e)]})

with gr.Blocks(title="日常支出速算與分攤") as demo:
    gr.Markdown("## 🧾 日常支出速算與分攤（Gradio）\n- 新增支出後自動寫回 Google Sheet\n- 一鍵查看總額、分類小計與 AA 分攤\n- 讀寫工作表：`工作表2`")
    tz = timezone(timedelta(hours=8))
    now_tw = datetime.datetime.now(tz)

    with gr.Tab("➕ 新增支出"):
        #快捷按鈕
        gr.Markdown("### ⚡ 快速填入")
        with gr.Row():
            btn_coffee = gr.Button("☕ 咖啡 $65")
            btn_lunch = gr.Button("🍱 雞腿便當 $120")
            btn_bus = gr.Button("🚌 交通 $30")

        #輸入欄位
        with gr.Row():
            date_in = gr.Textbox(label="日期", value=now_tw.strftime("%Y-%m-%d"))
            time_in = gr.Textbox(label="時間", value=now_tw.strftime("%H:%M"), placeholder="留白自動偵測")
        with gr.Row():
            cat_in = gr.Textbox(label="分類", placeholder="如 外食 / 交通 / 購物")
            item_in = gr.Textbox(label="品項", placeholder="如 咖啡 / 便當 / 車票")
        with gr.Row():
            amt_in = gr.Textbox(label="金額", placeholder="數字")
            payer_in = gr.Textbox(label="付款人", placeholder="如 Pecu / Alice / Bob")
        with gr.Row():
            loc_in = gr.Textbox(label="地點", placeholder="如 7-11 / 台北車站")
            note_in = gr.Textbox(label="備註", placeholder="如 幫同事代購 / 慶生")
        add_btn = gr.Button("新增到工作表")

        #定義快捷按鈕邏輯
        def quick_fill(item, amt, cat):
            return cat, item, str(amt)

        btn_coffee.click(fn=lambda: quick_fill("咖啡", 65, "餐飲"), outputs=[cat_in, item_in, amt_in])
        btn_lunch.click(fn=lambda: quick_fill("雞腿便當", 120, "餐飲"), outputs=[cat_in, item_in, amt_in])
        btn_bus.click(fn=lambda: quick_fill("公車", 30, "交通"), outputs=[cat_in, item_in, amt_in])

        add_msg = gr.Markdown()
        total_out = gr.Number(label="目前總額", interactive=False)
        cat_df = gr.Dataframe(label="分類小計", interactive=False)
        settle_df = gr.Dataframe(label="AA 分攤結算", interactive=False)

        add_btn.click(
            fn=add_expense,
            inputs=[date_in, time_in, cat_in, item_in, amt_in, payer_in, loc_in, note_in],
            outputs=[add_msg, total_out, cat_df, settle_df]
        )

    with gr.Tab("📊 彙總 / AA 分攤"):
        with gr.Row():
            # 設定每月預算
            budget_limit = gr.Number(label="💰 每月預算目標", value=15000)
            refresh_btn = gr.Button("🔄 讀取最新彙總", variant="primary")

        # 智慧進度條放在最上方
        budget_progress_html = gr.HTML()

        msg2 = gr.Markdown()

        with gr.Row():
            total2 = gr.Number(label="本月總支出", interactive=False)

        with gr.Row():
            cat_df2 = gr.Dataframe(label="分類小計", interactive=False)
            settle_df2 = gr.Dataframe(label="AA 分攤結算", interactive=False)

        raw_preview = gr.Dataframe(label="（預覽）最近資料", interactive=False)

        # 綁定點擊事件
        refresh_btn.click(
            fn=refresh_summary,
            inputs=[budget_limit],
            outputs=[msg2, total2, cat_df2, settle_df2, raw_preview, budget_progress_html]
        )

    with gr.Tab("📒 檢視原始資料"):
        view_btn = gr.Button("🔄 讀取最新資料與分析", variant="primary")

        # 新增 Plot 元件放圓餅圖
        view_chart = gr.Plot(label="支出佔比圖")

        # 表格元件
        view_df = gr.Dataframe(
            label="全部資料清單",
            interactive=False,
            wrap=True # 備註太長會自動換行
        )

        # 綁定點擊事件：一個按鈕觸發後更新「圖表」與「表格」兩個元件
        view_btn.click(
            fn=_view_all_with_chart,
            inputs=[],
            outputs=[view_chart, view_df]
        )
#預算進度條
def get_budget_progress(current_total, limit):
    if limit <= 0: return "<p>請設定預算目標</p>"
    percent = (current_total / limit) * 100
    display_percent = min(percent, 100) # 畫條最高到 100%

    # 顏色邏輯
    color = "#4CAF50" # 綠色
    if percent >= 100: color = "#F44336" # 紅色
    elif percent >= 80: color = "#FF9800" # 橘色

    html = f"""
    <div style="width: 100%; background-color: #e0e0e0; border-radius: 12px; margin: 15px 0; overflow: hidden; box-shadow: inset 0 1px 3px rgba(0,0,0,0.2);">
        <div style="width: {display_percent}%; background-color: {color}; height: 30px; text-align: center; line-height: 30px; color: white; font-weight: bold; transition: width 0.8s ease-in-out;">
            {percent:.1f}%
        </div>
    </div>
    <div style="display: flex; justify-content: space-between; font-size: 14px; margin-top: -10px;">
        <span>已花費: ${current_total:,.0f}</span>
        <span>剩餘預算: <b style="color:{color};">${max(0, limit - current_total):,.0f}</b></span>
    </div>
    """
    return html

# 啟動介面
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://65bc5a87f139dca0a0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
